<a href="https://colab.research.google.com/github/srijabiswas-01/banking-fraud-intelligence/blob/main/banking_fraud_network_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install neo4j networkx pandas matplotlib

In [ ]:
from neo4j import GraphDatabase
from getpass import getpass

URI = input("Enter Neo4j Aura URI: ").strip()
USERNAME = input("Enter Neo4j username [neo4j]: ").strip() or "neo4j"
PASSWORD = getpass("Enter Neo4j password: ")

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

driver.verify_connectivity()

print("Successfully connected to Neo4j AuraDB!")

Enter Neo4j Aura URI: neo4j+s://da4b13c2.databases.neo4j.io
Enter Neo4j username [neo4j]: da4b13c2
Enter Neo4j password: ··········
Successfully connected to Neo4j AuraDB!


In [ ]:
with driver.session() as session:
    result = session.run("""
        MATCH (a:Account)
        RETURN count(a) AS total_accounts
    """)

    print("Total Accounts:", result.single()["total_accounts"])

Total Accounts: 10


In [ ]:
import pandas as pd

query = """
MATCH (a1:Account)-[r:TRANSFERRED_TO]->(a2:Account)

RETURN
    a1.account_id AS source,
    a2.account_id AS target,
    r.total_amount AS amount,
    r.transaction_count AS transactions
"""

with driver.session() as session:
    records = session.run(query).data()

df = pd.DataFrame(records)

print("Total Transfer Relationships:", len(df))

display(df.head(10))

Total Transfer Relationships: 15


,source,target,amount,transactions
0,A010,A001,3000,1
1,A001,A002,2500,1
2,A005,A003,90000,1
3,A003,A004,95000,1
4,A004,A005,92000,1
5,A001,A006,5000,1
6,A002,A006,3500,1
7,A002,A007,6500,1
8,A006,A007,1800,1
9,A003,A008,85000,1


In [ ]:
import networkx as nx

G = nx.DiGraph()

# Include every bank account, including isolated accounts.
with driver.session() as session:
    accounts = session.run("""
        MATCH (a:Account)
        RETURN a.account_id AS account_id
    """).data()

G.add_nodes_from(row["account_id"] for row in accounts)

for _, row in df.iterrows():
    G.add_edge(
        row["source"],
        row["target"],
        weight=float(row["amount"]),
        transactions=int(row["transactions"])
    )

print("Total Nodes:", G.number_of_nodes())
print("Total Edges:", G.number_of_edges())

Total Nodes: 10
Total Edges: 15


In [ ]:
G_undirected = G.to_undirected()

components = list(
    nx.connected_components(G_undirected)
)

print("Total Connected Components:", len(components))

for i, component in enumerate(components, start=1):
    print(f"Component {i}: {sorted(component)}")

Total Connected Components: 1
Component 1: ['A001', 'A002', 'A003', 'A004', 'A005', 'A006', 'A007', 'A008', 'A009', 'A010']


In [ ]:
communities = nx.community.louvain_communities(
    G_undirected,
    weight=None,
    seed=42
)

community_mapping = {}

for community_id, community in enumerate(communities):
    for account in community:
        community_mapping[account] = community_id

print("Total Communities:", len(communities))

for i, community in enumerate(communities):
    print(f"Community {i}: {sorted(community)}")

Total Communities: 2
Community 0: ['A001', 'A002', 'A006', 'A007', 'A010']
Community 1: ['A003', 'A004', 'A005', 'A008', 'A009']


In [ ]:
community_df = pd.DataFrame(
    [
        {
            "Account_ID": account,
            "Community_ID": community_id
        }
        for account, community_id in community_mapping.items()
    ]
)

display(
    community_df.sort_values(
        ["Community_ID", "Account_ID"]
    )
)

,Account_ID,Community_ID
0,A001,0
2,A002,0
1,A006,0
3,A007,0
4,A010,0
7,A003,1
8,A004,1
6,A005,1
9,A008,1
5,A009,1


In [ ]:
pagerank_scores = nx.pagerank(
    G,
    alpha=0.85,
    weight=None
)

pagerank_df = pd.DataFrame(
    [
        {
            "Account_ID": account,
            "PageRank": score
        }
        for account, score in pagerank_scores.items()
    ]
)

pagerank_df = pagerank_df.sort_values(
    "PageRank",
    ascending=False
)

display(pagerank_df)

,Account_ID,PageRank
7,A008,0.173913
8,A009,0.173913
9,A010,0.137799
0,A001,0.132132
6,A007,0.131428
5,A006,0.101398
1,A002,0.071157
4,A005,0.026087
2,A003,0.026087
3,A004,0.026087


In [ ]:
!pip -q install plotly networkx pandas

In [ ]:
import plotly.graph_objects as go
import networkx as nx
import pandas as pd
import numpy as np

In [ ]:
from neo4j import GraphDatabase

# Check the default database associated with your connection
with driver.session() as session:
    result = session.run("""
        MATCH (n)
        RETURN count(n) AS total_nodes
    """)

    print("Total Nodes:", result.single()["total_nodes"])

Total Nodes: 54


In [ ]:
import pandas as pd

# Retrieve account risk information from Neo4j
query = """
MATCH (a:Account)

RETURN
    a.account_id AS account_id,
    coalesce(a.risk_score, 0) AS risk_score,
    coalesce(a.risk_level, 'Unknown') AS risk_level,
    coalesce(a.community_id, -1) AS community_id
"""

# Connect to the default database
with driver.session() as session:

    account_data = session.run(query).data()

# Convert to Pandas DataFrame
account_df = pd.DataFrame(account_data)

print("Account data retrieved successfully!")

print("Total Accounts:", len(account_df))

display(account_df)

Account data retrieved successfully!
Total Accounts: 10


,account_id,risk_score,risk_level,community_id
0,A001,0,Low,-1
1,A002,0,Low,-1
2,A003,100,High,-1
3,A004,100,High,-1
4,A005,100,High,-1
5,A006,0,Low,-1
6,A007,0,Low,-1
7,A008,25,Low,-1
8,A009,25,Low,-1
9,A010,0,Low,-1


In [ ]:
# Prepare community and PageRank results

results = []

for account in G.nodes():

    results.append({
        "account_id": account,
        "community_id": int(community_mapping[account]),
        "pagerank": float(pagerank_scores[account])
    })

# Cypher query to update Neo4j
write_query = """
UNWIND $rows AS row

MATCH (a:Account {account_id: row.account_id})

SET a.community_id = row.community_id,
    a.pagerank = row.pagerank

RETURN count(a) AS updated_accounts
"""

# Write results to the default database
with driver.session() as session:

    result = session.run(
        write_query,
        rows=results
    )

    print(
        "Updated Accounts:",
        result.single()["updated_accounts"]
    )

Updated Accounts: 10


In [ ]:
with driver.session() as session:

    result = session.run("""
        SHOW CURRENT USER
        YIELD home
        RETURN home
    """)

    print(result.single())

<Record home='da4b13c2'>


Extract Fraud Indicators from Neo4j

In [ ]:
import pandas as pd
import numpy as np

query = """
MATCH (c:Customer)-[:OWNS]->(a:Account)

OPTIONAL MATCH (c)-[:USES]->(d:Device)<-[:USES]-(other:Customer)
WHERE other <> c

WITH c, a,
     count(DISTINCT other) AS shared_device_customers

OPTIONAL MATCH (c)-[:REGISTERED_AT]->(addr:Address)
               <-[:REGISTERED_AT]-(other2:Customer)
WHERE other2 <> c

WITH c, a, shared_device_customers,
     count(DISTINCT other2) AS shared_address_customers

OPTIONAL MATCH (a)-[:INITIATED]->(t:Transaction)

WITH c, a,
     shared_device_customers,
     shared_address_customers,
     count(DISTINCT t) AS outgoing_transactions,
     count(DISTINCT CASE
         WHEN t.amount >= 50000 THEN t
     END) AS high_value_transactions

RETURN
    c.customer_id AS customer_id,
    a.account_id AS account_id,
    shared_device_customers,
    shared_address_customers,
    outgoing_transactions,
    high_value_transactions,
    coalesce(a.community_id, -1) AS community_id,
    coalesce(a.pagerank, 0.0) AS pagerank
"""

with driver.session() as session:
    records = session.run(query).data()

fraud_df = pd.DataFrame(records)

display(fraud_df)

,customer_id,account_id,shared_device_customers,shared_address_customers,outgoing_transactions,high_value_transactions,community_id,pagerank
0,C001,A001,0,0,2,0,0,0.132132
1,C002,A002,0,0,2,0,0,0.071157
2,C003,A003,2,2,2,2,1,0.026087
3,C004,A004,2,2,2,2,1,0.026087
4,C005,A005,2,2,2,2,1,0.026087
5,C006,A006,0,0,1,0,0,0.101398
6,C007,A007,0,0,1,0,0,0.131428
7,C008,A008,1,1,1,1,1,0.173913
8,C009,A009,1,1,1,1,1,0.173913
9,C010,A010,0,0,1,0,0,0.137799


Retrieve Circular Transfer Accounts

In [ ]:
cycle_query = """
MATCH
(a1:Account)-[:INITIATED]->(t1:Transaction)-[:SENT_TO]->
(a2:Account)-[:INITIATED]->(t2:Transaction)-[:SENT_TO]->
(a3:Account)-[:INITIATED]->(t3:Transaction)-[:SENT_TO]->
(a1)

WHERE a1 <> a2
  AND a2 <> a3
  AND a1 <> a3
  AND t1.timestamp < t2.timestamp
  AND t2.timestamp < t3.timestamp
  AND t3.timestamp <= t1.timestamp + duration({minutes:30})

RETURN DISTINCT
    a1.account_id AS account_1,
    a2.account_id AS account_2,
    a3.account_id AS account_3
"""

with driver.session() as session:
    cycle_records = session.run(cycle_query).data()

cycle_accounts = set()

for row in cycle_records:
    cycle_accounts.update([
        row["account_1"],
        row["account_2"],
        row["account_3"]
    ])

print("Accounts involved in rapid circular transfers:")
print(sorted(cycle_accounts))

Accounts involved in rapid circular transfers:
['A003', 'A004', 'A005']


Calculate Risk Scores

In [ ]:
# Create indicator columns
fraud_df["shared_device_flag"] = (
    fraud_df["shared_device_customers"] >= 2
).astype(int)

fraud_df["shared_address_flag"] = (
    fraud_df["shared_address_customers"] >= 2
).astype(int)

fraud_df["high_value_flag"] = (
    fraud_df["high_value_transactions"] >= 1
).astype(int)

fraud_df["circular_transfer_flag"] = (
    fraud_df["account_id"].isin(cycle_accounts)
).astype(int)

# Top 20% of accounts by PageRank
pagerank_threshold = fraud_df["pagerank"].quantile(0.80)

fraud_df["high_pagerank_flag"] = (
    fraud_df["pagerank"] > pagerank_threshold
).astype(int)

# Calculate enhanced risk score
fraud_df["enhanced_risk_score"] = (
    fraud_df["shared_device_flag"] * 25
    + fraud_df["shared_address_flag"] * 15
    + fraud_df["high_value_flag"] * 20
    + fraud_df["circular_transfer_flag"] * 25
    + fraud_df["high_pagerank_flag"] * 15
)

display(
    fraud_df[
        [
            "account_id",
            "community_id",
            "pagerank",
            "enhanced_risk_score"
        ]
    ].sort_values(
        "enhanced_risk_score",
        ascending=False
    )
)

,account_id,community_id,pagerank,enhanced_risk_score
3,A004,1,0.026087,85
2,A003,1,0.026087,85
4,A005,1,0.026087,85
7,A008,1,0.173913,35
8,A009,1,0.173913,35
0,A001,0,0.132132,0
1,A002,0,0.071157,0
5,A006,0,0.101398,0
6,A007,0,0.131428,0
9,A010,0,0.137799,0


Account Risk

In [ ]:
def classify_risk(score):
    if score >= 70:
        return "High"
    elif score >= 40:
        return "Medium"
    return "Low"

fraud_df["enhanced_risk_level"] = (
    fraud_df["enhanced_risk_score"].apply(classify_risk)
)

display(
    fraud_df[
        [
            "account_id",
            "enhanced_risk_score",
            "enhanced_risk_level"
        ]
    ].sort_values(
        "enhanced_risk_score",
        ascending=False
    )
)

,account_id,enhanced_risk_score,enhanced_risk_level
3,A004,85,High
2,A003,85,High
4,A005,85,High
7,A008,35,Low
8,A009,35,Low
0,A001,0,Low
1,A002,0,Low
5,A006,0,Low
6,A007,0,Low
9,A010,0,Low


Write Enhanced Risk Scores Back to Neo4j

In [ ]:
rows = fraud_df[
    [
        "account_id",
        "enhanced_risk_score",
        "enhanced_risk_level"
    ]
].to_dict("records")

write_query = """
UNWIND $rows AS row

MATCH (a:Account {account_id: row.account_id})

SET
    a.enhanced_risk_score = row.enhanced_risk_score,
    a.enhanced_risk_level = row.enhanced_risk_level

RETURN count(a) AS updated_accounts
"""

with driver.session() as session:
    result = session.run(
        write_query,
        rows=rows
    )

    print(
        "Updated Accounts:",
        result.single()["updated_accounts"]
    )

Updated Accounts: 10


In [ ]:
import plotly.express as px

risk_counts = (
    fraud_df["enhanced_risk_level"]
    .value_counts()
    .reindex(["Low", "Medium", "High"], fill_value=0)
    .reset_index()
)

risk_counts.columns = ["Risk Level", "Accounts"]

fig_risk = px.bar(
    risk_counts,
    x="Risk Level",
    y="Accounts",
    color="Risk Level",
    text="Accounts",
    color_discrete_map={
        "Low": "#10B981",
        "Medium": "#F59E0B",
        "High": "#EF4444"
    },
    title="Banking Fraud Risk Distribution"
)

fig_risk.update_layout(
    template="plotly_dark",
    height=500,
    showlegend=False,
    xaxis_title="Risk Classification",
    yaxis_title="Number of Accounts"
)

fig_risk.update_traces(
    textposition="outside"
)

fig_risk.show()

In [ ]:
community_risk = (
    fraud_df.groupby("community_id", as_index=False)
    .agg(
        Total_Accounts=("account_id", "count"),
        Average_Risk_Score=("enhanced_risk_score", "mean"),
        High_Risk_Accounts=(
            "enhanced_risk_level",
            lambda x: (x == "High").sum()
        )
    )
)

display(community_risk)

,community_id,Total_Accounts,Average_Risk_Score,High_Risk_Accounts
0,0,5,0.0,0
1,1,5,65.0,3


In [ ]:
fig_community = px.bar(
    community_risk,
    x="community_id",
    y="Average_Risk_Score",
    color="High_Risk_Accounts",
    text="Average_Risk_Score",
    color_continuous_scale="Reds",
    title="Average Fraud Risk by Louvain Community",
    hover_data=["Total_Accounts"]
)

fig_community.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside"
)

fig_community.update_layout(
    template="plotly_dark",
    height=550,
    xaxis_title="Louvain Community",
    yaxis_title="Average Risk Score"
)

fig_community.show()

In [ ]:
pagerank_plot = fraud_df.sort_values(
    "pagerank",
    ascending=True
)

fig_pagerank = px.bar(
    pagerank_plot,
    x="pagerank",
    y="account_id",
    orientation="h",
    color="enhanced_risk_level",
    color_discrete_map={
        "Low": "#10B981",
        "Medium": "#F59E0B",
        "High": "#EF4444"
    },
    hover_data=[
        "enhanced_risk_score",
        "community_id"
    ],
    title="Account Influence and Fraud Risk"
)

fig_pagerank.update_layout(
    template="plotly_dark",
    height=600,
    xaxis_title="PageRank Score",
    yaxis_title="Bank Account",
    legend_title="Risk Level"
)

fig_pagerank.show()

In [ ]:
from pathlib import Path
import plotly.io as pio

# --------------------------------------------------
# BANKING FRAUD INTELLIGENCE DASHBOARD
# OFFLINE INTERACTIVE HTML VERSION
# --------------------------------------------------

# Check that all charts exist
charts = {
    "Fraud Risk Distribution": fig_risk,
    "Community Risk Analysis": fig_community,
    "PageRank Account Ranking": fig_pagerank
}

# HTML header
dashboard_html = """
<!DOCTYPE html>
<html lang="en">

<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">

<title>Banking Fraud Intelligence Dashboard</title>

<style>

* {
    box-sizing: border-box;
}

body {
    background: #0B1120;
    color: #F8FAFC;
    font-family: Arial, sans-serif;
    margin: 0;
    padding: 35px;
}

.header {
    text-align: center;
    margin-bottom: 35px;
}

.header h1 {
    font-size: 32px;
    margin-bottom: 10px;
}

.header p {
    color: #94A3B8;
    font-size: 15px;
}

.dashboard-container {
    max-width: 1450px;
    margin: auto;
}

.chart-card {
    background: #111827;
    border: 1px solid #1E293B;
    border-radius: 14px;
    padding: 20px;
    margin-bottom: 25px;
}

.chart-card h2 {
    font-size: 19px;
    margin-bottom: 15px;
    color: #E2E8F0;
}

.footer {
    text-align: center;
    color: #64748B;
    margin-top: 35px;
    font-size: 13px;
}

</style>

</head>

<body>

<div class="dashboard-container">

<div class="header">

<h1>Banking Fraud Intelligence Dashboard</h1>

<p>
Neo4j Graph Analytics | Louvain Communities |
PageRank | Enhanced Risk Scoring
</p>

</div>
"""

# --------------------------------------------------
# EMBED PLOTLY JAVASCRIPT
# --------------------------------------------------

for index, (title, chart) in enumerate(charts.items()):

    chart_html = pio.to_html(
        chart,
        full_html=False,
        include_plotlyjs=True if index == 0 else False,
        config={
            "responsive": True,
            "displaylogo": False,
            "scrollZoom": True
        }
    )

    dashboard_html += f"""
    <div class="chart-card">
        <h2>{title}</h2>
        {chart_html}
    </div>
    """

# --------------------------------------------------
# HTML FOOTER
# --------------------------------------------------

dashboard_html += """

<div class="footer">

Banking Fraud Detection & Suspicious Transaction
Network Analysis

<br><br>

Synthetic Data | Graph Analytics Portfolio Project

</div>

</div>

</body>
</html>
"""

# --------------------------------------------------
# SAVE HTML FILE
# --------------------------------------------------

output_file = "Banking_Fraud_Dashboard.html"

Path(output_file).write_text(
    dashboard_html,
    encoding="utf-8"
)

print("Dashboard successfully generated!")
print("File:", output_file)

print(
    "File size:",
    round(Path(output_file).stat().st_size / 1024 / 1024, 2),
    "MB"
)

Dashboard successfully generated!
File: Banking_Fraud_Dashboard.html
File size: 4.38 MB


In [ ]:
from google.colab import files

files.download("Banking_Fraud_Dashboard.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

query = """
MATCH (a:Account)

RETURN
    a.account_id AS account_id,
    a.community_id AS community_id,
    a.pagerank AS pagerank,
    a.enhanced_risk_score AS risk_score,
    a.enhanced_risk_level AS risk_level

ORDER BY risk_score DESC
"""

with driver.session() as session:
    records = session.run(query).data()

verification_df = pd.DataFrame(records)

print("Total Accounts:", len(verification_df))

display(verification_df)

Total Accounts: 10


,account_id,community_id,pagerank,risk_score,risk_level
0,A003,1,0.026087,85,High
1,A004,1,0.026087,85,High
2,A005,1,0.026087,85,High
3,A008,1,0.173913,35,Low
4,A009,1,0.173913,35,Low
5,A001,0,0.132132,0,Low
6,A002,0,0.071157,0,Low
7,A006,0,0.101398,0,Low
8,A007,0,0.131428,0,Low
9,A010,0,0.137799,0,Low


In [ ]:
print("DATABASE VERIFICATION REPORT")
print("-" * 40)

print("Total Accounts:", len(verification_df))

print(
    "Missing Community IDs:",
    verification_df["community_id"].isna().sum()
)

print(
    "Missing PageRank Scores:",
    verification_df["pagerank"].isna().sum()
)

print(
    "Missing Risk Scores:",
    verification_df["risk_score"].isna().sum()
)

print(
    "Missing Risk Classifications:",
    verification_df["risk_level"].isna().sum()
)

DATABASE VERIFICATION REPORT
----------------------------------------
Total Accounts: 10
Missing Community IDs: 0
Missing PageRank Scores: 0
Missing Risk Scores: 0
Missing Risk Classifications: 0


In [ ]:
!pip -q install streamlit plotly neo4j pandas networkx

In [ ]:
import streamlit
import plotly
import neo4j
import networkx

print("Streamlit:", streamlit.__version__)
print("Plotly:", plotly.__version__)
print("Neo4j:", neo4j.__version__)
print("NetworkX:", networkx.__version__)

print("\nAll dashboard libraries installed successfully!")

Streamlit: 1.66.1
Plotly: 5.24.1
Neo4j: 6.4.1
NetworkX: 3.7

All dashboard libraries installed successfully!


In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import plotly.express as px
from neo4j import GraphDatabase

# --------------------------------------------------
# PAGE CONFIGURATION
# --------------------------------------------------

st.set_page_config(
    page_title="Banking Fraud Intelligence",
    page_icon="🏦",
    layout="wide",
    initial_sidebar_state="expanded"
)

# --------------------------------------------------
# CUSTOM DASHBOARD DESIGN
# --------------------------------------------------

st.markdown("""
<style>

.stApp {
    background-color: #0B1120;
}

.block-container {
    padding-top: 2rem;
    padding-bottom: 2rem;
}

h1, h2, h3 {
    color: #F8FAFC;
}

[data-testid="stMetric"] {
    background-color: #111827;
    border: 1px solid #1E293B;
    padding: 20px;
    border-radius: 12px;
}

[data-testid="stMetricLabel"] {
    color: #94A3B8;
}

[data-testid="stMetricValue"] {
    color: #F8FAFC;
}

</style>
""", unsafe_allow_html=True)

# --------------------------------------------------
# NEO4J CONNECTION
# --------------------------------------------------

@st.cache_resource
def get_driver():

    uri = st.secrets["NEO4J_URI"]
    username = st.secrets["NEO4J_USERNAME"]
    password = st.secrets["NEO4J_PASSWORD"]

    driver = GraphDatabase.driver(
        uri,
        auth=(username, password)
    )

    driver.verify_connectivity()

    return driver


# --------------------------------------------------
# LOAD ACCOUNT DATA
# --------------------------------------------------

@st.cache_data(ttl=300)
def load_accounts():

    query = """
    MATCH (a:Account)

    RETURN
        a.account_id AS account_id,
        coalesce(a.community_id, -1) AS community_id,
        coalesce(a.pagerank, 0.0) AS pagerank,
        coalesce(a.enhanced_risk_score, 0) AS risk_score,
        coalesce(a.enhanced_risk_level, 'Unknown') AS risk_level

    ORDER BY risk_score DESC
    """

    with get_driver().session() as session:
        records = session.run(query).data()

    return pd.DataFrame(records)


# --------------------------------------------------
# LOAD TRANSACTION DATA
# --------------------------------------------------

@st.cache_data(ttl=300)
def load_transactions():

    query = """
    MATCH (a1:Account)-[:INITIATED]->(t:Transaction)
          -[:SENT_TO]->(a2:Account)

    RETURN
        a1.account_id AS source,
        a2.account_id AS target,
        t.transaction_id AS transaction_id,
        t.amount AS amount
    """

    with get_driver().session() as session:
        records = session.run(query).data()

    return pd.DataFrame(records)


# --------------------------------------------------
# DASHBOARD HEADER
# --------------------------------------------------

st.title("Banking Fraud Intelligence Dashboard")

st.caption(
    "Neo4j Graph Analytics | Louvain Communities | "
    "PageRank | Enhanced Risk Scoring"
)

st.divider()

# --------------------------------------------------
# LOAD DATABASE DATA
# --------------------------------------------------

try:
    accounts = load_accounts()
    transactions = load_transactions()

except Exception as e:
    st.error("Unable to load banking data from Neo4j.")
    st.exception(e)
    st.stop()


if accounts.empty:
    st.warning("No account records were found.")
    st.stop()

# --------------------------------------------------
# EXECUTIVE KPI METRICS
# --------------------------------------------------

total_accounts = len(accounts)

total_transactions = len(transactions)

total_value = (
    pd.to_numeric(
        transactions["amount"],
        errors="coerce"
    ).sum()
    if not transactions.empty else 0
)

high_risk_accounts = (
    accounts["risk_level"] == "High"
).sum()

col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric(
        "Total Accounts",
        f"{total_accounts:,}"
    )

with col2:
    st.metric(
        "Total Transactions",
        f"{total_transactions:,}"
    )

with col3:
    st.metric(
        "Transaction Value",
        f"₹{total_value:,.0f}"
    )

with col4:
    st.metric(
        "High-Risk Accounts",
        f"{high_risk_accounts:,}"
    )

st.divider()

# --------------------------------------------------
# RISK DISTRIBUTION
# --------------------------------------------------

left, right = st.columns(2)

with left:

    st.subheader("Account Risk Distribution")

    risk_counts = (
        accounts["risk_level"]
        .value_counts()
        .reset_index()
    )

    risk_counts.columns = [
        "Risk Level",
        "Accounts"
    ]

    fig_risk = px.bar(
        risk_counts,
        x="Risk Level",
        y="Accounts",
        color="Risk Level",
        text="Accounts",
        color_discrete_map={
            "Low": "#10B981",
            "Medium": "#F59E0B",
            "High": "#EF4444",
            "Unknown": "#64748B"
        }
    )

    fig_risk.update_layout(
        template="plotly_dark",
        paper_bgcolor="#111827",
        plot_bgcolor="#111827",
        showlegend=False,
        height=400
    )

    st.plotly_chart(
        fig_risk,
        use_container_width=True
    )

# --------------------------------------------------
# PAGERANK ACCOUNT ANALYSIS
# --------------------------------------------------

with right:

    st.subheader("Account Network Influence")

    ranking = accounts.sort_values(
        "pagerank",
        ascending=True
    )

    fig_rank = px.bar(
        ranking,
        x="pagerank",
        y="account_id",
        orientation="h",
        color="risk_level",
        color_discrete_map={
            "Low": "#10B981",
            "Medium": "#F59E0B",
            "High": "#EF4444",
            "Unknown": "#64748B"
        },
        hover_data=["risk_score", "community_id"]
    )

    fig_rank.update_layout(
        template="plotly_dark",
        paper_bgcolor="#111827",
        plot_bgcolor="#111827",
        height=400,
        xaxis_title="PageRank",
        yaxis_title="Account",
        legend_title="Risk Level"
    )

    st.plotly_chart(
        fig_rank,
        use_container_width=True
    )

# --------------------------------------------------
# ACCOUNT INVESTIGATION TABLE
# --------------------------------------------------

st.divider()

st.subheader("Account Risk Investigation")

st.dataframe(
    accounts,
    use_container_width=True,
    hide_index=True
)

# --------------------------------------------------
# FOOTER
# --------------------------------------------------

st.divider()

st.caption(
    "Banking Fraud Detection & Suspicious Transaction "
    "Network Analysis | Synthetic Demonstration Dataset"
)

Writing app.py


In [ ]:
from pathlib import Path
from getpass import getpass
import json

# Create Streamlit configuration directory
secrets_dir = Path(".streamlit")
secrets_dir.mkdir(exist_ok=True)

# Database connection details
uri = "neo4j+s://da4b13c2.databases.neo4j.io"
username = "neo4j"

# Secure password input
password = getpass("Enter Neo4j database password: ")

# Create Streamlit secrets file
secrets_content = (
    f"NEO4J_URI = {json.dumps(uri)}\n"
    f"NEO4J_USERNAME = {json.dumps(username)}\n"
    f"NEO4J_PASSWORD = {json.dumps(password)}\n"
)

secrets_path = secrets_dir / "secrets.toml"

secrets_path.write_text(
    secrets_content,
    encoding="utf-8"
)

# Restrict file access
secrets_path.chmod(0o600)

del password
del secrets_content

print("Streamlit secrets configured successfully.")

Enter Neo4j database password: ··········
Streamlit secrets configured successfully.


In [ ]:
import subprocess
import time

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "app.py",
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true"
    ],
    stdout=open("/tmp/streamlit.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("Streamlit server started.")
print("Process ID:", process.pid)

Streamlit server started.
Process ID: 6357


In [ ]:
%%writefile requirements.txt
streamlit
neo4j
pandas
plotly
networkx
numpy

Writing requirements.txt


In [ ]:
from google.colab import files

files.download("app.py")
files.download("requirements.txt")
files.download(".gitignore")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
%%writefile .gitignore
.streamlit/secrets.toml
.env
.env.*
__pycache__/
*.pyc
.ipynb_checkpoints/

Writing .gitignore
